# MafutaPlan: Component-Based Multiple Linear Regression

**Project title:** Design and Implementation of a Component-Based
Fuel Price Prediction System Using Multiple Linear Regression in
Nairobi, Kenya

This notebook keeps four ideas separate:

1. **Prediction:** pre-target components estimate the following cycle.
2. **Reconstruction:** same-cycle official components are added.
3. **Scenario analysis:** user assumptions are added deterministically.
4. **Fuel calculations:** prices are converted into budgets and journeys.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_component_history, load_prediction_dataset
from src.modeling import JULY_2026_CYCLE, evaluate_latest_cycle
from src.pricing import reconstruction_audit

components = load_component_history()
model_data = load_prediction_dataset()
evaluation = evaluate_latest_cycle(model_data)

## Data integrity

Every component row includes an official EPRA HTTPS source, a
verification status, and a reconstruction result. No missing component
is imputed. The model-ready table links each input cycle to the
following target cycle.

In [2]:
coverage = pd.DataFrame({
    "Measure": [
        "Verified component rows",
        "Verified component cycles",
        "Model-ready rows",
        "Earliest input cycle",
        "Latest input cycle",
    ],
    "Value": [
        len(components),
        components["Effective_From"].nunique(),
        len(model_data),
        model_data["Input_Cycle"].min().date(),
        model_data["Input_Cycle"].max().date(),
    ],
})
coverage

,Measure,Value
0,Verified component rows,33
1,Verified component cycles,11
2,Model-ready rows,33
3,Earliest input cycle,2024-08-01
4,Latest input cycle,2026-03-01


In [3]:
audit = reconstruction_audit(components)
audit[["Effective_From", "Fuel", "Retail_Price",
       "Calculated_Price", "Calculated_Error"]].tail(9)

,Effective_From,Fuel,Retail_Price,Calculated_Price,Calculated_Error
24,2026-01-15,Diesel,170.47,170.47,0.0
25,2026-01-15,Kerosene,153.78,153.78,0.0
26,2026-01-15,Super Petrol,182.52,182.52,0.0
27,2026-02-15,Diesel,166.54,166.54,0.0
28,2026-02-15,Kerosene,152.78,152.78,0.0
29,2026-02-15,Super Petrol,178.28,178.28,0.0
30,2026-03-15,Diesel,166.54,166.54,0.0
31,2026-03-15,Kerosene,152.78,152.78,0.0
32,2026-03-15,Super Petrol,178.28,178.28,0.0


## Chronological multiple linear regression

The pooled model uses landed cost, distribution and storage, margins,
stabilization adjustment, taxes and levies, plus encoded fuel type.
The latest complete target cycle is reserved as a chronological test.

In [4]:
summary = pd.DataFrame({
    "Training start": [evaluation.training_start.date()],
    "Training end": [evaluation.training_end.date()],
    "Training rows": [evaluation.training_records],
    "Test target": [evaluation.test_cycle.date()],
    "Test rows": [evaluation.test_records],
    "MAE (KSh/L)": [evaluation.mae],
    "RMSE (KSh/L)": [evaluation.rmse],
})
summary

,Training start,Training end,Training rows,Test target,Test rows,MAE (KSh/L),RMSE (KSh/L)
0,2024-09-01,2026-03-01,30,2026-04-01,3,15.410728,19.299255


In [5]:
evaluation.coefficients

,Term,Coefficient
0,Intercept,134.108520
1,Landed_Cost,0.526436
2,Distribution_Storage,-61.973967
3,Margins,8.945869
4,Stabilization_Adjustment,0.300605
5,Taxes_Levies,1.737948
6,Fuel_Diesel,-9.775039
7,Fuel_Kerosene,-2.348368


In [6]:
evaluation.results

,Input_Cycle,Target_Cycle,Fuel,Target_Retail_Price,Predicted_Retail_Price,Absolute_Error,Percentage_Error
0,2026-03-01,2026-04-01,Diesel,196.63,167.857747,28.772253,14.632687
1,2026-03-01,2026-04-01,Kerosene,152.78,153.229972,0.449972,0.294523
2,2026-03-01,2026-04-01,Super Petrol,197.60,180.590043,17.009957,8.608278


## July 2026 availability decision

July prices must not be predicted from July components. The intended
input is June 2026. The reviewed component panel currently stops at
March 2026, so the notebook does not publish July predictions or
accuracy values. This is a data limitation, not a software failure.

In [7]:
july_rows = model_data.loc[model_data["Target_Cycle"].eq(JULY_2026_CYCLE)]
pd.DataFrame({
    "Required target": ["July 2026"],
    "Intended input": ["June 2026 components"],
    "Verified July-input rows": [len(july_rows)],
    "Prediction status": [
        "Blocked: verified June 2026 components are unavailable"
        if len(july_rows) != 3 else "Available"
    ],
})

,Required target,Intended input,Verified July-input rows,Prediction status
0,July 2026,June 2026 components,0,Blocked: verified June 2026 components are una...


## Interpretation

The small, discontinuous panel limits generalisation. The April 2026
holdout includes an abrupt regulatory price change, producing large
errors for petrol and diesel. Coefficients describe this fitted sample
and should not be interpreted as causal effects. MafutaPlan remains an
academic tool and does not replace EPRA.